In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/long-running-durable/lra-gcp/solutions")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 05 · A model-chosen tool loop, and Mistral as the provider

In notebooks 01–03, the model writes text and the workflow sets the order of the steps. In this notebook, the model selects the *next action*: a tool call or a final answer. Thus the model call becomes a non-deterministic side effect (primer §1.3). If you ask again after a crash, it is possible that the model selects a different call. The solution is to **write the decision to the journal before you act on it**. Then act at most one time, under a stable idempotency key.

In the second half of the notebook, Mistral becomes the model provider (offline, with a fake client). The second half also shows the same loop on Mistral Workflows. There, the platform gives the journal, the retries and the wait.

In [2]:
from datetime import timedelta
import json
from lra import Engine, Event, Budget, Workflow, Next, Done, Wait, StepFailed, RunStatus, StepTask, SimulatedCrash
from lra.adapters.memory import FakeClock, FakeLLM, InMemoryStateStore, InMemoryTaskQueue, InMemoryEventBus, LocalRunner
from lra.examples import ALL_WORKFLOWS
from lra.examples.scripted import research_routes

def harness(routes=None, fail_times=0, chaos=None, lease_ttl_s=60):
    """One engine over in-memory adapters, driven the way Cloud Tasks would drive it."""
    clock, store, bus = FakeClock(), InMemoryStateStore(), InMemoryEventBus()
    queue = InMemoryTaskQueue(clock)
    llm = FakeLLM(routes=routes or research_routes(), fail_times=fail_times)
    engine = Engine(store=store, queue=queue, bus=bus, llm=llm, clock=clock, workflows=ALL_WORKFLOWS,
                    worker_id="worker-A", lease_ttl=timedelta(seconds=lease_ttl_s), chaos=chaos)
    return engine, LocalRunner(engine, queue, clock), clock, store, queue, bus

def show(run):
    print(f"{run.run_id} {run.status.value:12s} step={run.current_step} attempts={run.attempts} "
          f"wait={run.wait.key if run.wait else None} steps_used={run.budget.steps_used}")

In [3]:
from lra.examples.tool_agent import PaymentAPI, ScriptedDecider, make_tool_agent

def tool_harness(wf, chaos=None):
    clock, store, bus = FakeClock(), InMemoryStateStore(), InMemoryEventBus()
    queue = InMemoryTaskQueue(clock)
    engine = Engine(store=store, queue=queue, bus=bus, llm=FakeLLM(), clock=clock, workflows=[wf],
                    worker_id="worker-A", lease_ttl=timedelta(seconds=60), chaos=chaos)
    return engine, LocalRunner(engine, queue, clock), clock, store

def show_journal(run):
    for e in run.state.get("journal", []):
        print("   ", e)

## Your turn: the two effect records of a turn

`decide` records the decision of the model with `ctx.effect` *before* anything acts on it. `act` runs the tool at most one time. It gives the tool an idempotency key (`run_id:N`, where N is the position of the intent in the journal). The key is stable across retries and unique across runs. Fill in the two keys.

In [4]:
def build_loop(decider, tools):
    wf = Workflow("my_loop", default_budget=Budget(max_steps=10))

    @wf.step(start=True)
    def decide(ctx):
        journal = ctx.state.setdefault("journal", [])
        n = len(journal)
        d = ctx.effect(f"decide:{n}", lambda: decider.decide(ctx.input["goal"], journal))
        journal.append({"type": "decision", **d})
        if "final" in d:
            return Done(d["final"])
        journal.append({"type": "intent", "tool": d["tool"], "args": d["args"], "key": f"{ctx.run_id}:{n + 1}", "done": False})
        return Next("act")

    @wf.step()
    def act(ctx):
        journal = ctx.state["journal"]
        intent, n = journal[-1], len(journal) - 1
        intent["result"] = ctx.effect(f"act:{n}", lambda: tools[intent["tool"]](**intent["args"], key=intent["key"]))
        intent["done"] = True
        return Next("decide")

    return wf

# A model that would choose differently if asked again, and a crash right after the first decision.
crashed = []
def chaos(point, run):
    if point == "after_step_before_commit" and run.current_step == "decide" and not crashed:
        crashed.append(run.run_id); raise SimulatedCrash()

pay = PaymentAPI()
decider = ScriptedDecider([{"tool": "charge", "args": {"amount": 42}}, {"tool": "charge", "args": {"amount": 99}}, {"final": "done"}])
engine, runner, clock, store = tool_harness(build_loop(decider, {"charge": pay}), chaos=chaos)
run = engine.start("my_loop", {"goal": "pay invoice 42"})
try:
    runner.run_until_idle()
except SimulatedCrash:
    print("crashed after the model decided, before the checkpoint")
clock.advance(seconds=61); engine.reap(); runner.run_until_idle()
r = store.get(run.run_id); show(r); show_journal(r)
print("charges:", pay.charges, "| model calls:", decider.calls)
assert pay.charges == {f"{run.run_id}:1": 42.0, f"{run.run_id}:3": 99.0} and decider.calls == 3

crashed after the model decided, before the checkpoint
run_fc96f3e1184f SUCCEEDED    step=None attempts={'decide': 3, 'act': 2} wait=None steps_used=5
    {'type': 'decision', 'tool': 'charge', 'args': {'amount': 42}}
    {'type': 'intent', 'tool': 'charge', 'args': {'amount': 42}, 'key': 'run_fc96f3e1184f:1', 'done': True, 'result': {'charge_id': 'run_fc96f3e1184f:1', 'amount': 42.0}}
    {'type': 'decision', 'tool': 'charge', 'args': {'amount': 99}}
    {'type': 'intent', 'tool': 'charge', 'args': {'amount': 99}, 'key': 'run_fc96f3e1184f:3', 'done': True, 'result': {'charge_id': 'run_fc96f3e1184f:3', 'amount': 99.0}}
    {'type': 'decision', 'final': 'done'}
charges: {'run_fc96f3e1184f:1': 42.0, 'run_fc96f3e1184f:3': 99.0} | model calls: 3


If there is no decision record, the retry asks the model again and acts on a different answer. Here, that answer is `99` in place of the `42` that the model selected first. Then the world and the run do not agree about the decision of the agent. With the decision record, the `decide` step asked the model three times in all. The answers were `42` (recorded one time and used again by the retry), then `99`, then `final`.

## Crash after the charge

The library version, `lra.examples.tool_agent.make_tool_agent`, is the same loop with an approval gate added. Cause a crash immediately after the charge. The retry executes *the same intent under the same key* again, finds the effect record and never asks the model again.

In [5]:
crashed = []
def chaos(point, run):
    if point == "after_step_before_commit" and run.current_step == "act" and not crashed:
        crashed.append(run.run_id); raise SimulatedCrash()

pay, decider = PaymentAPI(), ScriptedDecider([{"tool": "charge", "args": {"amount": 42}}, {"final": "Charged 42."}])
engine, runner, clock, store = tool_harness(make_tool_agent(decider, {"charge": pay}, gated=()), chaos=chaos)
run = engine.start("tool_agent", {"goal": "pay invoice 42"})
try:
    runner.run_until_idle()
except SimulatedCrash:
    print("crashed after the charge, before the checkpoint | charges so far:", pay.charges)
clock.advance(seconds=61); engine.reap(); runner.run_until_idle()
r = store.get(run.run_id); show(r)
print("charges:", pay.charges, "| payment API calls:", pay.calls, "| model calls:", decider.calls)
assert pay.calls == 1 and decider.calls == 2 and r.status == RunStatus.SUCCEEDED

crashed after the charge, before the checkpoint | charges so far: {'run_05a420c64815:1': 42.0}
run_05a420c64815 SUCCEEDED    step=None attempts={'decide': 2, 'act': 1} wait=None steps_used=3
charges: {'run_05a420c64815:1': 42.0} | payment API calls: 1 | model calls: 2


## When the model gets it wrong

Two types of mistake are for the model to correct, not for the engine. The first is a tool name that the model made up. The second is a tool that answers with a declared, non-retryable error (`ToolError`: incorrect arguments, no such invoice). The loop writes either one to the journal as the result of the call, and asks the model again. The run does not fail. Any other exception (a timeout, a 503) is an infrastructure problem, and the engine retries the step under the same key.

In [6]:
from lra.examples.tool_agent import ToolError

def lookup(invoice, key):
    if invoice != "INV-1042":
        raise ToolError(f"no invoice {invoice}")
    return {"invoice": invoice, "amount": 42}

pay = PaymentAPI()
decider = ScriptedDecider([{"tool": "refund", "args": {"amount": 42}},           # a tool that does not exist
                           {"tool": "lookup", "args": {"invoice": "INV-1024"}},  # a typo in the argument
                           {"tool": "lookup", "args": {"invoice": "INV-1042"}},
                           {"tool": "charge", "args": {"amount": 42}}, {"final": "Charged 42."}])
engine, runner, clock, store = tool_harness(make_tool_agent(decider, {"charge": pay, "lookup": lookup}, gated=()))
run = engine.start("tool_agent", {"goal": "pay invoice INV-1042"})
runner.run_until_idle()
r = store.get(run.run_id); show(r); show_journal(r)
assert r.status == RunStatus.SUCCEEDED and decider.calls == 5
assert r.state["journal"][1]["result"] == {"error": "unknown tool 'refund'"}
assert r.state["journal"][3]["result"] == {"error": "no invoice INV-1024"}

run_e18e44f4f99d SUCCEEDED    step=None attempts={'decide': 5, 'act': 4} wait=None steps_used=9
    {'type': 'decision', 'tool': 'refund', 'args': {'amount': 42}}
    {'type': 'intent', 'tool': 'refund', 'args': {'amount': 42}, 'key': 'run_e18e44f4f99d:1', 'done': True, 'result': {'error': "unknown tool 'refund'"}}
    {'type': 'decision', 'tool': 'lookup', 'args': {'invoice': 'INV-1024'}}
    {'type': 'intent', 'tool': 'lookup', 'args': {'invoice': 'INV-1024'}, 'key': 'run_e18e44f4f99d:3', 'done': True, 'result': {'error': 'no invoice INV-1024'}}
    {'type': 'decision', 'tool': 'lookup', 'args': {'invoice': 'INV-1042'}}
    {'type': 'intent', 'tool': 'lookup', 'args': {'invoice': 'INV-1042'}, 'key': 'run_e18e44f4f99d:5', 'done': True, 'result': {'invoice': 'INV-1042', 'amount': 42}}
    {'type': 'decision', 'tool': 'charge', 'args': {'amount': 42}}
    {'type': 'intent', 'tool': 'charge', 'args': {'amount': 42}, 'key': 'run_e18e44f4f99d:7', 'done': True, 'result': {'charge_id': 'run_

## Approve what you execute

A gated tool makes the run wait, with the *exact* proposed call in the journal. An approval executes that call. A rejection becomes the result of the tool, so the model can select again. A double click is a no-op.

In [7]:
pay = PaymentAPI()
decider = ScriptedDecider([{"tool": "charge", "args": {"amount": 4200}}, {"final": "Not paid: vendor not onboarded."}])
engine, runner, clock, store = tool_harness(make_tool_agent(decider, {"charge": pay}))
run = engine.start("tool_agent", {"goal": "pay invoice 4200"})
runner.run_until_idle()
r = store.get(run.run_id); show(r); show_journal(r)
reject = Event(run_id=run.run_id, key=r.wait.key, payload={"decision": "reject", "by": "cfo", "comment": "vendor not onboarded"})
print("first:", engine.resume(reject) is not None, "| second:", engine.resume(reject))
runner.run_until_idle()
r = store.get(run.run_id); show(r); show_journal(r)
assert pay.charges == {} and r.result == {"result": "Not paid: vendor not onboarded."}

run_4d44a736ae3f WAITING      step=act attempts={'decide': 1} wait=tool-1:run_4d44a736ae3f steps_used=1
    {'type': 'decision', 'tool': 'charge', 'args': {'amount': 4200}}
    {'type': 'intent', 'tool': 'charge', 'args': {'amount': 4200}, 'key': 'run_4d44a736ae3f:1', 'done': False}
first: True | second: None
run_4d44a736ae3f SUCCEEDED    step=None attempts={'decide': 2, 'act': 1} wait=None steps_used=3
    {'type': 'decision', 'tool': 'charge', 'args': {'amount': 4200}}
    {'type': 'intent', 'tool': 'charge', 'args': {'amount': 4200}, 'key': 'run_4d44a736ae3f:1', 'done': True, 'result': {'rejected': 'vendor not onboarded'}}
    {'type': 'decision', 'final': 'Not paid: vendor not onboarded.'}


## Swap the model: Mistral decides, by function calling

`lra.adapters.mistral.MistralDecider` changes the journal into a chat. The chat has the goal first. Then it has every earlier decision as an assistant tool call with its recorded result. The adapter then asks for the next call with `tool_choice="auto"`.

The fake client below uses the response shape of the SDK, so this cell runs offline. With `pip install -e ".[mistral]"` and `MISTRAL_API_KEY`, `MistralDecider(TOOL_SPECS)` calls the real API (`python -m lra.adapters.mistral.demo live`). A tool-call id for Mistral must have exactly nine alphanumeric characters, so the adapter hashes the journal key.

In [8]:
from types import SimpleNamespace as NS
from lra.adapters.mistral import DEFAULT_INSTRUCTIONS, TOOL_SPECS, MistralDecider, MistralLLM, to_messages, tool_call_id

class FakeMistral:
    """Scripted chat completions in the SDK's response shape; keeps every request."""
    def __init__(self, script):
        self.script, self.requests, self.chat = list(script), [], self
    def complete(self, **kw):
        self.requests.append(kw)
        item = self.script.pop(0)
        if "tool" in item:
            msg = NS(content="", tool_calls=[NS(id="abc123def", type="function",
                                                  function=NS(name=item["tool"], arguments=json.dumps(item["args"])))])
        else:
            msg = NS(content=item["final"], tool_calls=None)
        return NS(choices=[NS(message=msg)], usage=NS(prompt_tokens=900, completion_tokens=40))

fake = FakeMistral([{"tool": "charge", "args": {"amount": 42}}, {"final": "Charged 42."}])
pay = PaymentAPI()
engine, runner, clock, store = tool_harness(make_tool_agent(MistralDecider(TOOL_SPECS, model="mistral-small-latest", client=fake), {"charge": pay}))
run = engine.start("tool_agent", {"goal": "Pay supplier invoice INV-1042: 42 SGD."})
runner.run_until_idle()
r = store.get(run.run_id)
engine.resume(Event(run_id=run.run_id, key=r.wait.key, payload={"decision": "approve", "by": "cfo"}))
runner.run_until_idle()
r = store.get(run.run_id); show(r)
print("second prompt:", [m["role"] for m in fake.requests[1]["messages"]])
print("the tool result the model saw:", fake.requests[1]["messages"][-1]["content"])
assert r.status == RunStatus.SUCCEEDED and pay.charges == {f"{run.run_id}:1": 42.0}

run_c8d5ff3f5aaf SUCCEEDED    step=None attempts={'decide': 2, 'act': 1} wait=None steps_used=3
second prompt: ['system', 'user', 'assistant', 'tool']
the tool result the model saw: {"charge_id": "run_c8d5ff3f5aaf:1", "amount": 42.0}


## Your turn: the journal as the prompt

Build `to_messages` again. Each decision becomes an assistant tool call. The intent that comes immediately after the decision in the journal becomes the tool message for that call. The content of that message is the recorded result of the intent, or `{"pending": true}` if the intent has not run. Fill in the tool-call id and the content of the tool message.

In [9]:
def my_to_messages(goal, journal):
    messages = [{"role": "system", "content": DEFAULT_INSTRUCTIONS}, {"role": "user", "content": goal}]
    for i, step in enumerate(journal):
        if step["type"] != "decision" or "final" in step:
            continue
        intent = journal[i + 1]
        cid = tool_call_id(intent["key"])
        messages.append({"role": "assistant", "content": "", "tool_calls": [
            {"id": cid, "type": "function", "function": {"name": step["tool"], "arguments": json.dumps(step["args"])}}]})
        messages.append({"role": "tool", "tool_call_id": cid, "name": step["tool"], "content": json.dumps(intent.get("result") if intent["done"] else {"pending": True})})
    return messages

journals = [
    [],
    r.state["journal"],
    [{"type": "decision", "tool": "charge", "args": {"amount": 1}},
     {"type": "intent", "tool": "charge", "args": {"amount": 1}, "key": "r:1", "done": False}],
    [{"type": "decision", "tool": "charge", "args": {"amount": 5}},
     {"type": "intent", "tool": "charge", "args": {"amount": 5}, "key": "r:1", "done": True, "result": {"rejected": "no"}},
     {"type": "decision", "final": "stopped"}],
]
for j in journals:
    assert my_to_messages("goal", j) == to_messages("goal", j), j
print("my_to_messages matches the adapter on", len(journals), "journals; ids look like", tool_call_id("r:1"))

my_to_messages matches the adapter on 4 journals; ids look like 369714a8a


## Mistral as the engine's model

`MistralLLM` implements the same `LLM` port as `GeminiLLM` and `FakeLLM`. Thus every workflow in `lra.examples` runs on it with no change. The usage becomes cost for the budget of the run (the prices in the adapter are illustrative).

In [10]:
fake = FakeMistral([{"final": "billing"}])
wf = Workflow("classify")
@wf.step(start=True)
def classify(ctx):
    return Done(ctx.llm("Classify this ticket as billing/technical/other: I was charged twice").text)
clock, store = FakeClock(), InMemoryStateStore(); queue = InMemoryTaskQueue(clock)
engine = Engine(store=store, queue=queue, bus=InMemoryEventBus(), llm=MistralLLM(model="mistral-small-latest", client=fake),
                clock=clock, workflows=[wf])
run = engine.start("classify", {}); LocalRunner(engine, queue, clock).run_until_idle()
r = store.get(run.run_id)
print(r.status.value, r.result, "| tokens:", r.budget.tokens_used, "| cost (illustrative prices): $%.6f" % r.budget.cost_usd)
assert r.result == "billing" and r.budget.tokens_used == 940

SUCCEEDED billing | tokens: 940 | cost (illustrative prices): $0.000440


## The same loop on Mistral Workflows

`lra/adapters/mistral/workflow.py` is this loop on Mistral Workflows, with Temporal below it. The infrastructure code goes away: there is no store, no queue, no lease and no reaper. In their place:

* The **event history** of the execution is the store.
* The platform records every **activity** (the model call and the charge) before the activity runs. The platform retries the activity on failure, and never runs the activity again after the activity is complete.
* `workflow.wait_condition(...)` and a **signal** are the approval gate, with a timeout.
* A loop bound in deterministic workflow code and `execution_timeout` are the budget.

Note one detail that is easy to miss. An *unexpected* exception in workflow code fails only the workflow **task**. The platform retries that task until you repair the code and deploy it again. To fail an execution on purpose, raise `WorkflowError`.

The Mistral Workflows loop needs Python 3.12–3.14 and the `mistral` extra (`mistralai-workflows` 3.15 declares `Requires-Python >=3.12,<3.15`, checked 2026-09-26, verify). The loop also needs a local Temporal dev server, which the SDK downloads on first use. `python -m lra.adapters.mistral.demo workflow` runs the approval, crash-after-charge and timeout scenarios. `tests/test_mistral_workflow.py` has a test for each of these scenarios. If you installed the extra, the cell below shows the gate.

In [11]:
import importlib.util, inspect, sys
if sys.version_info >= (3, 12) and importlib.util.find_spec("mistralai") and importlib.util.find_spec("mistralai.workflows"):
    from lra.adapters.mistral import workflow as mw
    print(inspect.getsource(mw.InvoiceAgent.run))
else:
    print("Mistral Workflows is not installed here (Python 3.12+ and the mistral extra); read lra/adapters/mistral/workflow.py.")

Mistral Workflows is not installed here (Python 3.12+ and the mistral extra); read lra/adapters/mistral/workflow.py.


## Takeaways

* A model-chosen action is two side effects. Record the *decision*, then the *act*, each under its own key.
* On every retry, the journal is the prompt. It contains recorded facts, never the memory of the model.
* The provider is one adapter (`MistralDecider`, `MistralLLM`). The invariants do not change. A durable-execution platform (Mistral Workflows, Temporal, Cloud Workflows) gives you the invariants, and you still pass idempotency keys downstream.